<a href="https://colab.research.google.com/github/zoha200/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract
**Track:** Machine Learning · **Phase:** Foundations · **Week:** 3

**My lane:** Lane 2 — Refresh / Content Opportunity Scoring

**Panel month used for all development below:** `month=2026-03` (mid-panel, per the assignment's own example).
**Sealed test month (never touched for label/feature logic):** `month=2026-06` (the `_sample` file — final month of the panel).

> A note on how this notebook is built: rather than guessing exact column names for a 78.8M-row gated
> warehouse I can't query directly, **Section 0** below asks the warehouse for its own schema first and
> stores the real column names in variables. Every query after that references those variables — so the
> SQL actually runs against whatever the real column names turn out to be, instead of me hard-coding a guess
> that silently breaks. If the auto-detect picks the wrong column for something lane-specific, there's an
> override cell right below it — check the printed schema and edit that cell.


## Setup — install + connect

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn
import duckdb, os, pandas as pd
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)


In [6]:
BASE = "hf://datasets/FlyRank/internship-warehouse"
PANEL_MONTH = "2026-03"
TABLES = {
    "dim_clients": f"{BASE}/dim_clients.parquet",
    "dim_content": f"{BASE}/dim_content.parquet",
    "fact_content_daily_performance": f"{BASE}/fact_content_daily_performance/month={PANEL_MONTH}/data_0.parquet",
    "fact_content_query_90d": f"{BASE}/fact_content_query_90d.parquet",
}
print("Connected. Panel month:", PANEL_MONTH)


Connected. Panel month: 2026-03


In [7]:
from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=HF_TOKEN)
for f in sorted(files):
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

## Section 0 — Discover the real schema
Not one of the four graded deliverables on its own — this is the prep step that makes Sections 2 and 3
trustworthy instead of guessed. `DESCRIBE` on a `read_parquet(...)` glob only reads Parquet footers, so
this is cheap even on the full fact table.

In [8]:
schemas = {}
for name, path in TABLES.items():
    schemas[name] = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()
    print(f"--- {name} ---")
    print(schemas[name].to_string(index=False))
    print()


--- dim_clients ---
        column_name column_type null  key default extra
     client_hash_id     VARCHAR  YES None    None  None
          is_active     BOOLEAN  YES None    None  None
     has_gsc_access     BOOLEAN  YES None    None  None
     has_ga4_access     BOOLEAN  YES None    None  None
     access_profile     VARCHAR  YES None    None  None
client_created_date        DATE  YES None    None  None
client_updated_date        DATE  YES None    None  None
     gsc_data_start        DATE  YES None    None  None
     ga4_data_start        DATE  YES None    None  None

--- dim_content ---
               column_name column_type null  key default extra
            client_hash_id     VARCHAR  YES None    None  None
           content_hash_id     VARCHAR  YES None    None  None
           keyword_hash_id     VARCHAR  YES None    None  None
               url_hash_id     VARCHAR  YES None    None  None
        keyword_char_count      BIGINT  YES None    None  None
       keyword_token_

In [10]:
# Auto-detect the key columns on the fact table from the schema above.
def find_col(cols, *keywords, exclude=()):
    for c in cols:
        lc = c.lower()
        if all(k in lc for k in keywords) and not any(x in lc for x in exclude):
            return c
    return None

fact_schema = schemas["fact_content_daily_performance"]
fact_cols = fact_schema["column_name"].tolist()

CLIENT_COL  = find_col(fact_cols, "client")
CONTENT_COL = find_col(fact_cols, "content")
DATE_COL    = find_col(fact_cols, "date", exclude=("update",))
CLICK_COL   = find_col(fact_cols, "click")
IMPR_COL    = find_col(fact_cols, "impress")
POS_COL     = find_col(fact_cols, "position") or find_col(fact_cols, "rank")

bool_candidates = fact_schema.loc[
    fact_schema["column_type"].str.upper().str.contains("BOOL", na=False), "column_name"
].tolist()
AVAIL_COL = bool_candidates[0] if bool_candidates else None

print("CLIENT_COL  =", CLIENT_COL)
print("CONTENT_COL =", CONTENT_COL)
print("DATE_COL    =", DATE_COL)
print("CLICK_COL   =", CLICK_COL)
print("IMPR_COL    =", IMPR_COL)
print("POS_COL     =", POS_COL)
print("AVAIL_COL   =", AVAIL_COL, " (boolean columns found on the fact table:", bool_candidates, ")")


CLIENT_COL  = client_hash_id
CONTENT_COL = content_hash_id
DATE_COL    = report_date
CLICK_COL   = gsc_clicks
IMPR_COL    = gsc_impressions
POS_COL     = gsc_sum_position
AVAIL_COL   = client_has_gsc  (boolean columns found on the fact table: ['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available'] )


In [13]:

# --- Manual override -----------------------------------------------------
# Auto-detect picked client_has_gsc for AVAIL_COL, but that's a per-client
# capability flag, not a per-day sync/availability flag. Use the real
# per-row availability flag instead.
AVAIL_TABLE = "fact_content_daily_performance"
AVAIL_COL = "gsc_data_available"

print("AVAIL_COL overridden to:", AVAIL_COL)

AVAIL_COL overridden to: gsc_data_available


## 1) The contract, in plain words

**What one row means for my lane (Refresh / Content Opportunity Scoring):**
One row is one pseudonymized content item's performance on one calendar day, for one client —
the grain of `fact_content_daily_performance`. My lane doesn't change that grain; it aggregates
many of these daily rows *up* to one row per (client, content item) as of a decision date, which
is what actually gets scored and ranked for refresh.

**Which table(s) I'll use:**
- `fact_content_daily_performance` — the daily signal (clicks, impressions, position) I roll up into
  monthly features.
- `dim_content` — content-level metadata (e.g. when a page was published) that doesn't change day
  to day, used for age-style features.
- `dim_clients` — only to read `gsc_data_start` / `ga4_data_start`, so I don't score a client for a
  month before their data actually starts.
- I am **not** using `fact_content_query_90d` this week — see the exclusion below.

**Time window:**
Development happens entirely on `month=2026-03` (mid-panel). `month=2026-06` (the `_sample` file,
the panel's final month) is treated as sealed — never touched for feature or label logic, per the
assignment's own warning.

**What I'd predict or rank (label / proxy):**
A ranking/scoring problem, not a single label: *"which content items are most worth refreshing
next?"* For the leakage demo in Section 3 I use a concrete proxy so the trap is checkable —
`declined_next = 1` if a content item's average daily clicks in **April** (`month=2026-04`) are
lower than its average daily clicks in **March** (`month=2026-03`). April is only ever read to
*build the label*, never as an input feature — that boundary is exactly what the trap in Section 3
deliberately breaks and then repairs.

**One thing I deliberately exclude:**
`fact_content_query_90d`. Its window is a fixed trailing 90 days over the *query* table, not aligned
to my monthly panel cut — for a content item scored at the end of March, that 90-day query window
can extend past the decision date and into data I wouldn't have yet in production. Rather than
reason carefully about a partial, possibly-leaky overlap this week, I'm cutting it entirely and
sticking to `fact_content_daily_performance` + `dim_content`, which respect the March cutoff cleanly.


## 2) Prove three facts with three small queries
All three run against `month=2026-03` only.

### Query 1 — Grain: one row really is (client, content item, day)

In [14]:
q1 = f"""
SELECT {CLIENT_COL}, {CONTENT_COL}, {DATE_COL}, COUNT(*) AS n_rows
FROM read_parquet('{TABLES["fact_content_daily_performance"]}')
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 10
"""
dup_rows = con.sql(q1).df()
print(f"Combinations of ({CLIENT_COL}, {CONTENT_COL}, {DATE_COL}) that appear more than once: {len(dup_rows)}")
dup_rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Combinations of (client_hash_id, content_hash_id, report_date) that appear more than once: 0


,client_hash_id,content_hash_id,report_date,n_rows


**Reading the result:** zero rows back means the grain holds exactly as claimed — one row per
(client, content item, day), no duplicates within the March partition.

### Query 2 — My slice's row count and date span

In [15]:
q2 = f"""
SELECT
    COUNT(*)                       AS n_rows,
    MIN({DATE_COL})                AS min_date,
    MAX({DATE_COL})                AS max_date,
    COUNT(DISTINCT {CLIENT_COL})   AS n_clients,
    COUNT(DISTINCT {CONTENT_COL})  AS n_content_items
FROM read_parquet('{TABLES["fact_content_daily_performance"]}')
"""
con.sql(q2).df()


,n_rows,min_date,max_date,n_clients,n_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


### Query 3 — Availability, filtered with `IS TRUE`

In [16]:
if AVAIL_COL is None:
    # Fall back to dim_clients if the fact table itself has no boolean availability flag —
    # re-run Section 0's DESCRIBE cell output for dim_clients and set AVAIL_COL / AVAIL_TABLE above.
    raise ValueError("No boolean column auto-detected — set AVAIL_COL/AVAIL_TABLE manually from the Section 0 schema printout.")

q3 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE {AVAIL_COL} IS TRUE) AS available_rows
FROM read_parquet('{TABLES["fact_content_daily_performance"]}')
"""
avail = con.sql(q3).df()
avail["pct_available"] = (avail["available_rows"] / avail["total_rows"] * 100).round(2)
avail


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows,pct_available
0,9841378,3611061,36.69


## 3) Five features + the leakage trap

### 3a — Build the March feature frame (one row per client × content item)
Every feature below is computed **only from `month=2026-03` and `dim_content` metadata that predates
March** — nothing from April or later touches this frame yet.

In [18]:
features_q = f"""
SELECT
    f.{CLIENT_COL}  AS client_id,
    f.{CONTENT_COL} AS content_id,
    AVG(f.{CLICK_COL})                                          AS f1_avg_clicks_march,
    AVG(f.{IMPR_COL})                                           AS f2_avg_impressions_march,
    SUM(f.{POS_COL})::DOUBLE / NULLIF(SUM(f.{IMPR_COL}), 0)     AS f3_avg_position_march,
    SUM(f.{CLICK_COL})::DOUBLE / NULLIF(SUM(f.{IMPR_COL}), 0)   AS f4_ctr_march,
    COUNT(*)                                                    AS n_days_observed_march
FROM read_parquet('{TABLES["fact_content_daily_performance"]}') f
WHERE f.{AVAIL_COL} IS TRUE
GROUP BY 1, 2
"""
feat = con.sql(features_q).df()
print(feat.shape)
feat.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 7)


,client_id,content_id,f1_avg_clicks_march,f2_avg_impressions_march,f3_avg_position_march,f4_ctr_march,n_days_observed_march
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,0.032258,29.000000,5.825362,0.001112,31
1,client_62f4a7e64f5e0096,content_ac8663da7484669a,0.000000,2.000000,5.941176,0.000000,17
2,client_62f4a7e64f5e0096,content_d49a012dcb924e31,0.000000,10.612903,5.136778,0.000000,31
3,client_62f4a7e64f5e0096,content_614baf2af4330bd7,0.032258,24.903226,4.818653,0.001295,31
4,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,0.000000,1.400000,4.285714,0.000000,10


In [20]:
print(schemas["dim_content"]["column_name"].tolist())

['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


In [21]:
# f5 — content age at the decision moment (end of March), from dim_content metadata.
content_cols = schemas["dim_content"]["column_name"].tolist()
CONTENT_ID_COL = find_col(content_cols, "content")
PUBLISH_COL = "content_created_date"
print("dim_content id col:", CONTENT_ID_COL, "| publish-date-like col:", PUBLISH_COL)

age_q = f"""
SELECT {CONTENT_ID_COL} AS content_id,
       DATE '2026-03-31' - CAST({PUBLISH_COL} AS DATE) AS f5_content_age_days
FROM read_parquet('{TABLES["dim_content"]}')
"""
age = con.sql(age_q).df()
feat = feat.merge(age, on="content_id", how="left")
feat.head()

dim_content id col: content_hash_id | publish-date-like col: content_created_date


,client_id,content_id,f1_avg_clicks_march,f2_avg_impressions_march,f3_avg_position_march,f4_ctr_march,n_days_observed_march,f5_content_age_days
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,0.032258,29.000000,5.825362,0.001112,31,47
1,client_62f4a7e64f5e0096,content_ac8663da7484669a,0.000000,2.000000,5.941176,0.000000,17,47
2,client_62f4a7e64f5e0096,content_d49a012dcb924e31,0.000000,10.612903,5.136778,0.000000,31,47
3,client_62f4a7e64f5e0096,content_614baf2af4330bd7,0.032258,24.903226,4.818653,0.001295,31,47
4,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,0.000000,1.400000,4.285714,0.000000,10,47


**Why each feature is knowable at the decision moment (end of March):**
- **f1 avg daily clicks (March)** — computed entirely from days already elapsed by the decision date; no future days included.
- **f2 avg daily impressions (March)** — same: a rollup of days that have already happened.
- **f3 avg position (March)** — Search Console position data is reported daily with a short, already-elapsed lag; nothing here depends on April.
- **f4 CTR (March)** — a ratio of two March-only aggregates (f1/f2's underlying sums), so it inherits the same "already happened" guarantee.
- **f5 content age in days** — `publish_date` is fixed at publication time, which is always before the March 31 decision point by construction.


### 3b — The trap: one label-derived column, on purpose
`declined_next` is the **label** (built from April, the outcome window) — legitimate to build, illegitimate
to also feed in as a *feature*. Below I build a quick score honestly first, then deliberately smuggle an
April-derived column into the feature set and watch the score jump toward perfect, then delete it.

In [22]:
# Build the label from April (month=2026-04) — outcome window only, never touched above.
APRIL_PATH = f"{BASE}/fact_content_daily_performance/month=2026-04/*.parquet"
label_q = f"""
SELECT {CLIENT_COL} AS client_id, {CONTENT_COL} AS content_id,
       AVG({CLICK_COL}) AS avg_clicks_april
FROM read_parquet('{APRIL_PATH}')
GROUP BY 1, 2
"""
april = con.sql(label_q).df()

data = feat.merge(april, on=["client_id", "content_id"], how="inner")
data["declined_next"] = (data["avg_clicks_april"] < data["f1_avg_clicks_march"]).astype(int)
print(data.shape, "| decline rate:", data["declined_next"].mean().round(3))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176737, 10) | decline rate: 0.276


In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["f1_avg_clicks_march", "f2_avg_impressions_march", "f3_avg_position_march", "f4_ctr_march"]
if "f5_content_age_days" in data.columns:
    honest_features.append("f5_content_age_days")

def quick_auc(df, feature_cols, label_col="declined_next"):
    d = df.dropna(subset=feature_cols + [label_col])
    X, y = d[feature_cols], d[label_col]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
    clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    return roc_auc_score(yte, clf.predict_proba(Xte)[:, 1])

honest_auc = quick_auc(data, honest_features)
print("Honest AUC (March-only features):", round(honest_auc, 3))


Honest AUC (March-only features): 0.871


In [24]:
# --- THE TRAP: smuggle in an April-derived column as a FEATURE, not just the label ---
data["LEAK_avg_clicks_april"] = data["avg_clicks_april"]  # future data, disguised as a feature

leaky_features = honest_features + ["LEAK_avg_clicks_april"]
leaky_auc = quick_auc(data, leaky_features)
print("Leaky AUC (March features + a raw April column):", round(leaky_auc, 3))
print(f"Jump: {honest_auc:.3f} -> {leaky_auc:.3f}")


Leaky AUC (March features + a raw April column): 0.998
Jump: 0.871 -> 0.998


In [25]:
# --- Delete the leak, keep the honest number ---
data = data.drop(columns=["LEAK_avg_clicks_april"])
print("Leaky column removed. Final reported score is the honest one:", round(honest_auc, 3))
assert "LEAK_avg_clicks_april" not in data.columns


Leaky column removed. Final reported score is the honest one: 0.871


**The lesson (notebook 02, replayed on real warehouse data):** the leaky version isn't a better model —
it's a model that was handed the answer. `avg_clicks_april` is almost the same quantity `declined_next` is
computed from, so any classifier trivially "solves" the problem by reading it back off. The honest AUC using
only March-knowable features is the real, defensible number; the leaky jump is a warning sign, not a win.

## 4) One named limitation of this slice

The warehouse is an **unbalanced panel** (`dim_clients.gsc_data_start` / `ga4_data_start` differ per
client). Some clients in the `n_clients` count from Query 2 may only have partial March coverage if
their GSC/GA4 connection started mid-month — Query 2's date span is the *panel's* span, not a guarantee
that every client has a full 31 days of it. A model trained on this March slice without checking
`gsc_data_start` per client risks quietly under-weighting newly onboarded clients simply because they
have fewer observed days, not because they perform differently.

## 5) Self-check

- [ ] Five plain-words contract answers are filled in (Section 1) and specific to Lane 2.
- [ ] Exactly three verification queries ran with their outputs visible (Section 2); the availability
      query uses `IS TRUE`.
- [ ] Five-feature frame built from `month=2026-03` only, each with an "available when?" line (Section 3a).
- [ ] The deliberate leak is shown jumping the score, then removed, with the honest number kept (Section 3b).
- [ ] One named limitation of the slice is stated (Section 4).
- [ ] `month=2026-06` (`_sample`) was never read for label or feature logic anywhere in this notebook.
- [ ] Notebook executed top-to-bottom and committed to `work/notebooks/w03_data_contract.ipynb`.
